---
layout: post
codemirror: true
title: Microblog API HW
categories: [Flask, Python, APIs]
lesson_language: Python
lesson_topic: Microblog API HW
lesson_part: interactive
lesson_type: lesson
permalink: /python/flask/microblog-api-hw
author: rex0dex
---

## Microblog API

Aaditya Prem (rex0dex). Microblog is the comment panel on OCS pages. The lesson has me use it first, then read the Flask code behind it, then show its posts on a page of my own. The code I read is in my team's fork of the OCS Flask backend (`usc-flask`): `api/microblog_api.py`, `model/microblog.py`, and `api/authorize.py`. Every line number below is from that copy. I compared it with OCS upstream: `model/microblog.py` is the same, and `microblog_api.py` only differs in checking `account_types.admin_role()` instead of the string `'Admin'`.

| Part | What I did |
| --- | --- |
| Popcorn 1 | Drafted my post, worked out what the request should look like from the code, and traced what the backend does with it. The observed values table gets filled in after I post. |
| REST Client | Wrote the two requests and ran the read against my local backend without signing in to see what the decorator does |
| Read the API | Every method's inputs, checks, and status codes with line numbers, plus which GET branch wins |
| Follow the model | The page path branch into `Topic` and `MicroBlog.get_by_topic`, and the create path |
| Popcorn 2 | Matched the five Spring parts and answered the two questions |
| Popcorn 3 | UI Builder settings for the public page route, the 200 / 401 / 404 cases, and the `userUid` warning |
| Continue | Traced the reply resource the same way and listed requests that break the API |

The Python cells below only make read-only `GET` requests to my local copy of the backend (port 8588) or check the logic of the code with small copies of it. None of them sign in or change data.

## Popcorn Hack 1: Post and Watch the Request

**My post** (one sentence, nothing personal, since posts are public):

> An API should check that the post has content and the user is signed in before it saves anything.

It is 97 characters, well under 280. It also turned out to describe this exact API: `@token_required()` checks that I am signed in before `post()` runs, and line 32 checks that there is content.

**Steps:** dev tools, **Network** tab, filter `microblog`, press **Microblog** at the top of the lesson page, post the sentence, then click the `POST` request.

**What the request should look like** (worked out from the code before posting, so I can check it):

* **URL:** `POST https://flask.opencodingsociety.com/api/microblog`. `createPost` in `assets/js/api/microblog.js` posts to `pythonURI + '/api/microblog'`.
* **Body:** `{"content": "...", "topicPath": "/python/flask/api/microblog", "data": {"channel": ...}}`. The panel (`_includes/microblog_foundation.html`) sends the page's permalink as `topicPath`.
* **Status:** `200`, not `201`. Line 72 is `return jsonify(created_microblog.read())` with no status code, so Flask uses 200.
* **Reply:** the new post from `MicroBlog.read()` (model line 81): `id`, `userId`, `userName`, `userUid`, `content`, `topicId`, `topicKey`, `topicPath`, `timestamp`, `updatedAt`, `characterCount`, plus whatever was in `data`.

**TODO: fill in after I post from my account.**

| What I looked for | Expected from the code | Observed in the Network tab |
| --- | --- | --- |
| URL | `https://flask.opencodingsociety.com/api/microblog` (POST) | TODO |
| `content` | my sentence above | TODO |
| `topicPath` | `/python/flask/api/microblog` | TODO |
| Status code | `200` | TODO |
| Returned `id` | the new row's id (any number) | TODO |

### What happened to my request

Following the request through the backend in order:

| Step | Where | What it does |
| --- | --- | --- |
| 1 | `main.py` line 107, `microblog_api.py` lines 12 and 556 | `app.register_blueprint(microblog_api)`. The blueprint has `url_prefix='/api'` and `/microblog` is registered to `MicroBlogAPI._CRUD`, so `POST /api/microblog` lands in `_CRUD.post`. |
| 2 | `microblog_api.py` line 21 then `authorize.py` lines 146-151 | `@token_required()` is just an alias that returns `auth_required(roles, allow_pending)`. |
| 3 | `authorize.py` lines 42-63 | Uses the Flask-Login session if there is one (line 42), otherwise reads the JWT cookie named by `JWT_TOKEN_NAME` (line 50) or an `Authorization` header (lines 52-56). No token returns **401** before my handler ever runs. |
| 4 | `authorize.py` lines 67-117 | Decodes the token, finds the `User` by `_uid` (line 68), rejects an old token after a password change (line 80, **401**), rejects an inactive account (line 112, **403**), then sets `g.current_user` (line 89). |
| 5 | `microblog_api.py` lines 24-36 | `post()` reads `g.current_user`, then checks the body (line 28), that `content` exists (line 32), and that it is 280 characters or less (line 35). Each failure is a **400**. |
| 6 | lines 39-57 | `topicPath` was sent and `topicId` was not, so it calls `Topic.get_or_create_for_page(page_path=..., page_title=..., allow_anonymous=True)`. |
| 7 | model lines 478-498 | `get_or_create_for_page` returns the existing topic from `Topic.get_by_page_path` (line 468), or builds a new `Topic` whose key comes from `_generate_page_key` (line 381) and saves it. |
| 8 | lines 61-68, model lines 47-79 | `MicroBlog(user_id=current_user.id, content=..., topic_id=..., data=...)`. `__init__` checks the 280 limit again (model line 57), then `create()` does `db.session.add(self)` and `commit()`. |
| 9 | line 72, model line 81 | `created_microblog.read()` turns the row into a dictionary and `jsonify` sends it back with **200**. That is where the `id` in my Network tab comes from. |

## Read Posts with REST Client

These go in an `.http` file in the backend repo, sent after `api/user_auth.http` with the `local` environment so the auth cookie is reused:

{% raw %}

```http
### Authenticated read - expect an object with microblogs and count
GET {{baseUrl}}/api/microblog?limit=5
Accept: application/json

### Authenticated validation check - expect 400 and no new post
POST {{baseUrl}}/api/microblog
Content-Type: application/json
Accept: application/json

{
  "content": ""
}
```

{% endraw %}

The lesson warns that if authentication fails first, the content check was never tested. I wanted to see that happen, so I sent the read to my local backend **without** signing in:

In [1]:
# Read-only GET requests to my local copy of the backend. No sign-in cookie is sent.
import json
import urllib.error
import urllib.request

BASE = "http://localhost:8588"

def get(path):
    """GET a route on my local backend and print the status code and the JSON reply."""
    try:
        with urllib.request.urlopen(BASE + path, timeout=5) as reply:
            status, body = reply.status, reply.read()
    except urllib.error.HTTPError as error:   # a 4xx or 5xx reply still has a JSON body
        status, body = error.code, error.read()
    print(f"GET {path} -> {status}")
    print(json.dumps(json.loads(body), indent=2))
    print()

get("/api/microblog?limit=5")

GET /api/microblog?limit=5 -> 401
{
  "message": "Authentication required. No session or token found.",
  "data": null,
  "error": "Unauthorized"
}



The 401 message is word for word lines 58-63 of `authorize.py`. The decorator answered, so `get()` never ran. The empty-content POST would get the same 401 without a cookie, which is why the lesson has me authenticate first.

| Request | No sign-in | After `user_auth.http` (from the code) |
| --- | --- | --- |
| `GET /api/microblog?limit=5` | **401** `Authentication required. No session or token found.` (ran it above) | **200** `{"microblogs": [...], "count": n}` (lines 104-107), at most 5 posts, newest first |
| `POST /api/microblog` with `"content": ""` | **401**, same message (the decorator runs first) | **400** `Content is required` (lines 32-33). `""` is falsy, so `not content` is True and nothing is saved. |

## Read api/microblog_api.py

**Registration:** `microblog_api = Blueprint('microblog_api', __name__, url_prefix='/api')` (line 12) and `api.add_resource(MicroBlogAPI._CRUD, '/microblog', endpoint='microblog_crud')` (line 556). All four CRUD methods have `@token_required()` (lines 21, 79, 111, 145), so every one of them can return 401 or 403 before its own code runs.

| Method | Inputs | Checks in order | Model call | Status codes |
| --- | --- | --- | --- | --- |
| `post` (line 22) | JSON `content`, optional `topicId`, `topicPath`, `data` | body exists (28), `content` (32), `len(content) > 280` (35), topic lookup (44-57) | `Topic.get_or_create_for_page`, `MicroBlog(...)`, `.create()`, `.read()` | 200, 400, 500 |
| `get` (line 80) | query `limit` (default 200), `search`, `topicId`, `pagePath`, `userId` | one `if/elif` chain picks one branch (90-103) | `search_content`, `get_by_topic`, `Topic.get_by_page_path`, `get_by_user`, or `get_all` | 200, 500 |
| `put` (line 112) | JSON `id`, optional `content`, `data` | body (117), `id` (121), post exists (126, **404**), owner or Admin (130, **403**) | `MicroBlog.get_by_id`, `.update()`, `.read()` | 200, 400, 403, 404, 500 |
| `delete` (line 146) | JSON `id` | body (151), `id` (155), post exists (160, **404**), owner or Admin (164, **403**) | `MicroBlog.get_by_id`, `.delete()` | 200 with a message, 400, 403, 404, 500 |

Things I noticed reading the real status codes instead of assuming:

* A successful create is **200**, not 201, and a delete is 200 with `{'message': 'MicroBlog post deleted successfully'}` (line 169), not 204.
* The ownership check is `microblog._user_id != current_user.id and role != 'Admin'`, so an Admin can edit or delete anyone's post.
* `put` has no 280 check of its own. `MicroBlog.update()` checks it (model line 114) and raises `ValueError`, which line 140 turns into a 400.
* `request.args.get('topicId', type=int)` gives `None` when the value isn't a number, so `?topicId=abc` doesn't error. It just falls through to the next branch.

**Which GET branch wins.** The branches are `if search` / `elif topic_id` / `elif page_path` / `elif user_id` / `else`, so only the first one that is set runs. The filters are never combined. I copied the order into a function to check some mixed requests:

In [2]:
# Same if/elif order as MicroBlogAPI._CRUD.get (microblog_api.py lines 83-103)

def as_int(value):
    """What request.args.get(name, type=int) does: a number, or None if it can't convert."""
    try:
        return int(value) if value is not None else None
    except ValueError:
        return None

def get_branch(args):
    limit = as_int(args["limit"]) if "limit" in args else 200
    if limit is None:   # a bad limit falls back to the default too
        limit = 200
    topic_id = as_int(args.get("topicId"))
    page_path = args.get("pagePath")
    user_id = as_int(args.get("userId"))
    search = args.get("search")
    if search:
        return f"MicroBlog.search_content({search!r}, {limit})"
    elif topic_id:
        return f"MicroBlog.get_by_topic({topic_id}, {limit})"
    elif page_path:
        return f"Topic.get_by_page_path({page_path!r}) then MicroBlog.get_by_topic(topic.id, {limit})"
    elif user_id:
        return f"MicroBlog.get_by_user({user_id}, {limit})"
    else:
        return f"MicroBlog.get_all({limit})"

requests = [
    {"limit": "5"},
    {"userId": "7", "search": "API"},
    {"pagePath": "/python/flask/api/microblog", "topicId": "3"},
    {"pagePath": "/python/flask/api/microblog", "userId": "7"},
    {"topicId": "abc", "userId": "7"},
    {"search": "", "userId": "7"},
]
for args in requests:
    print(args)
    print("   ->", get_branch(args))

{'limit': '5'}
   -> MicroBlog.get_all(5)
{'userId': '7', 'search': 'API'}
   -> MicroBlog.search_content('API', 200)
{'pagePath': '/python/flask/api/microblog', 'topicId': '3'}
   -> MicroBlog.get_by_topic(3, 200)
{'pagePath': '/python/flask/api/microblog', 'userId': '7'}
   -> Topic.get_by_page_path('/python/flask/api/microblog') then MicroBlog.get_by_topic(topic.id, 200)
{'topicId': 'abc', 'userId': '7'}
   -> MicroBlog.get_by_user(7, 200)
{'search': '', 'userId': '7'}
   -> MicroBlog.get_by_user(7, 200)


* `search` beats everything, even when `userId` is also sent, so "search my own posts" isn't possible with this route.
* `topicId` beats `pagePath`. The page path is only used when no topic id is sent.
* `topicId=abc` turns into `None`, so the request quietly becomes a `userId` request instead of an error.
* An empty `search=` is falsy, so it is skipped.

## Follow model/microblog.py

**The page path branch** (`GET /api/microblog?pagePath=/python/flask/api/microblog`):

| Boundary | Input | Returns |
| --- | --- | --- |
| Handler to `Topic.get_by_page_path` (model line 468) | the `pagePath` string | `Topic.query.filter_by(_page_path=page_path).first()`: a `Topic` object or `None` |
| `None` case (line 99) | no topic | `microblogs = []`, so the reply is **200** with `count: 0`, not a 404 |
| Handler to `MicroBlog.get_by_topic` (model line 293) | `topic.id`, `limit` | `filter_by(_topic_id=...)`, newest first by `_timestamp`, `.limit(limit)`, then a **list of dictionaries** (each post's `read()`) |
| Handler to client (lines 104-107) | that list | `{"microblogs": [...], "count": len(...)}` |

The `Topic` lookup happens first because posts don't store the page path. They store `_topic_id` (model line 30), so the path has to become a topic id before any posts can be found.

**The create path** (my Popcorn Hack 1 request):

| Boundary | Input | Returns |
| --- | --- | --- |
| `Topic.get_or_create_for_page` (model 478) | `page_path`, `page_title`, `allow_anonymous=True` | the existing or new `Topic`, or `None` on failure (the handler turns `None` into a 500, line 55) |
| `Topic.__init__` (model 351) | the path | sets `_page_key = self._generate_page_key(page_path)` (line 370) |
| `MicroBlog(...)` (model 47) | `user_id`, `content`, `topic_id`, `data` | an object not saved yet. Raises `ValueError` over 280 characters (line 57). |
| `.create()` (model 67) | none | `self` after `commit()`, or `None` on an `IntegrityError` (handler gives 500, line 70) |
| `.read()` (model 81) | none | the dictionary that becomes my JSON reply |

Checking the key and title the create handler would make for this lesson page, using the same lines as `_generate_page_key` (model lines 385-389) and the handler's title (line 49):

In [3]:
import re

def generate_page_key(page_path):
    """Copy of Topic._generate_page_key, model/microblog.py lines 381-389."""
    key = re.sub(r"[^a-zA-Z0-9\-_]", "_", page_path.strip("/"))
    key = re.sub(r"_+", "_", key)
    return key[:100].strip("_")

topic_path = "/python/flask/api/microblog"
print("page key:  ", generate_page_key(topic_path))
print("page title:", repr(topic_path.replace("/", " ").title()))   # microblog_api.py line 49

# Two different paths can end up with the same key
print(generate_page_key("/python/flask/api/microblog"), "vs", generate_page_key("/python/flask_api/microblog"))

page key:   python_flask_api_microblog
page title: ' Python Flask Api Microblog'
python_flask_api_microblog vs python_flask_api_microblog


The key matches what the lesson says: `python_flask_api_microblog`. Two things I didn't expect:

* The title starts with a space (`' Python Flask Api Microblog'`) because the leading `/` becomes a space before `.title()`.
* Only letters, digits, `-` and `_` survive, so `/python/flask_api/microblog` and `/python/flask/api/microblog` would both become `python_flask_api_microblog`. `_page_key` is `unique=True` (model line 325), so creating the second topic would fail on the unique key, `get_or_create_for_page` would return `None`, and the post would fail with the 500 on line 55. Real OCS paths use `/`, so it doesn't come up, but it's a collision the key function doesn't prevent.

## Popcorn Hack 2: Match the Spring Parts

| Spring Part | Its Job | Where Microblog Does It (in my fork) |
| --- | --- | --- |
| Entity | Describes the table | `class MicroBlog(db.Model)`, `model/microblog.py` line 15, with `__tablename__ = 'microblogs'` and `db.Column` fields on lines 26-40 |
| JPA Repository | Reads and saves rows | static methods on the same class: `get_by_id` (280), `get_all` (286), `get_by_topic` (293), `get_by_user` (300), `search_content` (307), plus `create` (67) and `delete` (268) |
| Service | Holds the rules | no separate file. The rules are in the handler: `Content is required` (line 33), the 280 limit (line 35), the owner-or-Admin check (lines 130 and 164) |
| Controller | Turns HTTP into method calls | `MicroBlogAPI._CRUD` in `api/microblog_api.py` (line 18), registered at `/api/microblog` (line 556) |
| Table Initializer | Makes sure the table exists | `db.create_all()` in `scripts/db_init.py` line 149 builds `microblogs` and `topics` from the models |

### My answers

These are what I'm posting in the Microblog panel on the lesson page.

**1. The 280 limit in two places.** `len(content) > 280` in the handler (line 35) protects the **request**: it rejects a long post right away with a clear 400 (`Content must be 280 characters or less`) before any topic is created or the database is touched. `db.String(280)` in the model (line 33) protects the **table**: it makes `_content` a `VARCHAR(280)` column, so the database itself has a size limit even for code that skips the handler. There is actually a third check in `MicroBlog.__init__` (line 57) and `update()` (line 114), which covers the seed data in `initMicroblogs()` and `PUT`, since `put` has no handler check.

The column limit depends on the database. The deployed backend uses MySQL (`mysql+pymysql`, `__init__.py` line 113), but locally it's SQLite (line 118), and SQLite doesn't enforce `VARCHAR` lengths. I checked that:

In [4]:
import sqlite3

# A table with the same column type db.String(280) creates
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE microblogs (id INTEGER PRIMARY KEY, _content VARCHAR(280) NOT NULL)")
db.execute("INSERT INTO microblogs (_content) VALUES (?)", ("x" * 500,))
saved = db.execute("SELECT length(_content) FROM microblogs").fetchone()[0]
print("Characters SQLite saved in a VARCHAR(280) column:", saved)

Characters SQLite saved in a VARCHAR(280) column: 500


SQLite saved all 500 characters, so on my local backend the handler and model checks are the only things enforcing 280. That's a good reason the check lives in the code and not only in the column.

**2. Where the ownership check goes in Spring.** The **Service** (for example a `MicroblogService`). "Only the owner or an Admin can edit a post" is a business rule, not HTTP. The controller would just turn `PUT /api/microblog` into something like `service.update(id, content, currentUser)`, the repository would only do `findById` and `save`, and the service would load the post, compare the owner with the current user, and throw a forbidden error that the controller sends back as 403. Here Flask does all three jobs on lines 125-137 of `put`.

## Popcorn Hack 3: Show Posts with UI Builder

`GET /api/microblog` won't work in UI Builder because it has `@token_required()` and the builder never sends sign-in cookies. The route to use is `TopicAPI._PageMicroblogs`, registered at `/microblog/page/<string:page_key>` (line 563). Its `get` (line 464) has no decorator.

**How it decides the status code** (lines 478-509):

| Response | Line | Reason |
| --- | --- | --- |
| **404** `Page topic not found` | 480-482 | `Topic.get_by_page_key(page_key)` found no topic. Nobody has posted on that page yet, because posting is what creates the topic. |
| **403** `This discussion is currently disabled` | 484-485 | the topic exists but `_is_active` is False (not in the lesson's table, but the code has it) |
| **401** `Authentication required to view this discussion` | 488-489 | the topic has `_allow_anonymous` False and there is no current user |
| **200** | 492-506 | `{"topic": ..., "microblogs": [...], "count": n, "canPost": ..., "userPostCount": ...}` with up to `limit` posts (default 20, line 476), from `topic.get_recent_posts` (model line 456) |

For this lesson page the answer should be **200** once anyone has posted. My Popcorn Hack 1 post sends `topicPath`, and the create handler passes `allow_anonymous=True` (line 50), so the topic it makes allows anonymous viewers. A 401 would only come from a topic an Admin made through `TopicAPI._CRUD.post`, where `allowAnonymous` defaults to `False` (line 368).

My local backend doesn't have this topic yet, so it shows the 404 case:

In [5]:
get("/api/microblog/page/python_flask_api_microblog")
get("/api/microblog/topics?pagePath=/python/flask/api/microblog")

GET /api/microblog/page/python_flask_api_microblog -> 404
{
  "message": "Page topic not found"
}

GET /api/microblog/topics?pagePath=/python/flask/api/microblog -> 404
{
  "message": "Topic not found for this page"
}



Both say the topic doesn't exist, and the second one is the public topics route (`TopicAPI._CRUD.get`, line 385) looking it up by path. On the deployed backend the same link should come back 200 after my post.

**Something odd about the optional sign-in.** Lines 468-473 try `from api.authorize import get_current_user` inside a bare `try/except: pass`. I checked which functions `authorize.py` actually defines:

In [6]:
import ast
from pathlib import Path

repo = Path.home() / "rex0dex" / "usc-flask"
tree = ast.parse((repo / "api" / "authorize.py").read_text())
print("Functions in api/authorize.py:", [node.name for node in tree.body if isinstance(node, ast.FunctionDef)])

lines = (repo / "api" / "microblog_api.py").read_text().splitlines()
for number in range(468, 474):
    print(number, lines[number - 1])

Functions in api/authorize.py: ['auth_required', 'token_required']
468            try:
469                # Try to get user from token if provided, but don't require it
470                from api.authorize import get_current_user
471                current_user = get_current_user()
472            except:
473                pass  # No auth provided, continue as anonymous


There is no `get_current_user`, so the import always fails, the `except` swallows it, and `current_user` is always `None`. On this route everyone counts as anonymous, even someone signed in: a topic without `allow_anonymous` always gives 401, `canPost` is always `False`, and `userPostCount` is always 0. It doesn't hurt UI Builder, which sends no cookies anyway, but it means the "public endpoint with optional auth" docstring isn't true. Upstream's `authorize.py` doesn't have it either.

### My UI Builder block

1. Sign in, open UI Builder, start from **Flask API starter**.
2. In **Data from your API**, change the link from `/api/jokes` to the page route and press **Test connection**.
3. Map the fields to the post fields `MicroBlog.read()` returns (model lines 89-101).

The settings I'm using, written the same way the template's `api_data` block is (`_projects/systems/ui-builder/data/ui_builder.yml`):

```yaml
- type: api_data
  settings:
    title: What students said about this lesson
    view: list
    limit: 5
    source:
      url: "https://flask.opencodingsociety.com/api/microblog/page/python_flask_api_microblog"
      list_path: microblogs
      title_path: userName
      text_path: content
      image_path: ""
      link_path: ""
      value_path: ""
      label_path: timestamp
```

`list_path` is `microblogs` because the 200 reply is an object, and the posts are the list inside it. I picked **List**, so pressing **Code** should show the list view from `blocks-project.js` (`listView`), which uses the `ocs__entity-list` class:

```html
<ul class="ocs__entity-list">
  <li><strong>userName</strong><br><span>content</span></li>
  <!-- one li per post, up to 5 -->
</ul>
```

What **Test connection** should say for each reply, from the UI Builder's messages:

| Reply | What UI Builder shows |
| --- | --- |
| 200 | `Connected in ... ms. Found n items.`, then the `userUid` warning below |
| 401 | `This route needs sign-in. The builder never sends sign-in cookies...` |
| 404 | `... says this route doesn't exist (404). Check the path.` Here that really means no topic yet, not a wrong path. |

**The `userUid` warning.** `api-data.js` flags any field named like `uid`, `useruid`, `email`, `password`, or `token` (its `PRIVATE` list), so a 200 from this route shows `Your API sends userUid to anyone who asks. Remove them from this route.` It's right. `MicroBlog.read()` puts `'userUid': self.user.uid` (model line 93) in every post, and `_PageMicroblogs` returns that same `read()` with no sign-in. The uid is the login name, so this public route gives a stranger every poster's username, and only the password is left to guess. Everything a stranger actually needs is `userName`, `content`, and `timestamp`. The fix would be a public version of `read()` for this route that leaves out `userUid` (and `userId`). I'm not changing the backend for this homework, but that's the change I would make.

## Continue by Reading the Repository: the Reply Resource

Same routine as the API hub: URL, handler, permissions, input checks, model call, response.

| Step | `MicroBlogAPI._Reply` |
| --- | --- |
| Registered URL | `/api/microblog/reply` (line 557) |
| Handlers | `post` (line 178) with `@token_required()`, `get` (line 216) with **no** decorator, so reading replies is public |
| Input checks (`post`) | body (183), `postId` or the older `microblogId` (187, 191, **400**), `content` (194, **400**), post exists (199, **404**) |
| Input checks (`get`) | `postId` query param as an int (218-220, **400**), post exists (221-223, **404**) |
| Model call | `microblog.add_reply(current_user.id, content)` (model line 140): checks 280 (line 142), appends a reply dictionary to `_data['replies']`, then `flag_modified(self, '_data')` so SQLAlchemy notices the JSON changed, then `commit()`. `get` calls `get_replies()` (model line 133). |
| Response | `post`: 200 with `message`, `reply`, and the whole updated `microblog`. `get`: 200 with `{"replies": [...], "count": n}` |

Replies aren't their own table. They live inside the post's `_data` JSON column, and a reply's `id` is just `len(replies) + 1`, so ids only count within one post.

The public `get` lets me test the reply checks without signing in:

In [7]:
get("/api/microblog/reply")
get("/api/microblog/reply?postId=1")

GET /api/microblog/reply -> 400
{
  "message": "postId query param is required"
}

GET /api/microblog/reply?postId=1 -> 404
{
  "message": "MicroBlog post not found"
}



Both match the code: no `postId` is a 400 from line 220, and post 1 doesn't exist on my local database, so it's the 404 from line 223.

### Requests that fail

| Request | Status | Why | How I know |
| --- | --- | --- | --- |
| `GET /api/microblog?limit=5` with no cookie | 401 | `auth_required` finds no session or token (`authorize.py` 58-63) | ran it above |
| `GET /api/microblog/page/python_flask_api_microblog` before anyone posts | 404 | no topic with that key (line 482) | ran it above |
| `GET /api/microblog/reply` | 400 | no `postId` (line 220) | ran it above |
| `PUT /api/microblog` on someone else's post, signed in as a normal user | 403 | `_user_id != current_user.id` and not Admin (line 130) | code only, it would change data |
| `POST /api/microblog` with `"content": 123` | 500 | `not 123` is False, so it passes line 32, then `len(123)` on line 35 raises `TypeError` outside the `try`, so Flask returns a server error instead of a 400 | code only |

The last one is worth fixing: line 32 could check `isinstance(content, str)`. A post of only spaces (`"   "`) also passes line 32 because a non-empty string is truthy, so it would be saved. `if not content or not content.strip()` would catch it.

## What I Learned

* A Flask handler does the jobs of Spring's controller and service at once: it reads the request, checks the rules, calls the model, and builds the response. That's simpler to read, but the rules can't be reused or tested apart from HTTP.
* The decorator runs first. A 401 tells me nothing about my handler's checks, so I have to authenticate before testing validation.
* An `if/elif` chain of filters means only one filter is ever used, and the order decides which. I have to read the order instead of assuming the filters combine.
* A limit belongs in the code and in the column. SQLite ignored `VARCHAR(280)`, so without the handler and model checks my local database would take any length.
* A public route has to be checked for what it returns, not just for whether it needs sign-in. `_PageMicroblogs` returns every poster's `userUid`, which UI Builder caught.
* Reading the real code found things the lesson doesn't mention: a create returns 200 not 201, there's a 403 for a disabled topic, the optional sign-in import never works, and a non-string `content` crashes with a 500.